# Enrich Court Authority Cards — Qwen3.5-35B-A3B + vLLM

Batch-enriches `court_authority_cards_v4.jsonl` with English legal RAG metadata.

This version is tuned for the observed Google Compute Engine / Colab GPU setup with an RTX PRO 6000 Blackwell-class 96 GB GPU:

- Uses `Qwen/Qwen3.5-35B-A3B` via vLLM offline inference.
- Uses deterministic, schema-constrained JSON generation.
- Disables Qwen thinking at chat-template render time.
- Forces Triton MoE kernels to avoid FlashInfer CUTLASS SM120 JIT failures on Blackwell.
- Uses checkpointed JSONL append output.
- Keeps deterministic auto-classification for trivial notification/cost/short paragraphs.
- Adds batch throughput logging, JSON validation, retry, and high-error-rate abort protection.

Target output: `/content/drive/MyDrive/swiss_law/artifacts/court_authority_cards_rag.jsonl`


## 1 · Environment setup

In [ ]:
import sys, os

IN_COLAB = 'google.colab' in sys.modules
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'Running in Colab: {IN_COLAB}')

if IN_COLAB:
    # Verify GPU
    import subprocess
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                            capture_output=True, text=True)
    print('GPU:', result.stdout.strip())

In [ ]:
# Fresh runtime strongly recommended before running this cell.
# Do not import vllm / transformers / torch / torchvision / PIL / numpy before this cell.
#
# The installer is intentionally conservative:
# - pins Pillow below 12 to avoid the PIL._typing/_Ink mismatch seen in Colab-style runtimes
# - reinstalls NumPy/SciPy together to avoid mixed binary wheels
# - installs vLLM through uv with the official torch backend resolver
#
# If packages are already healthy and installed, set RUN_INSTALL = False.

import os
import sys
import site
import shutil
import subprocess
from pathlib import Path

RUN_INSTALL = IN_COLAB  # set True manually outside Colab if needed

def run(cmd, *, check=True):
    print("+", " ".join(map(str, cmd)))
    return subprocess.run(cmd, check=check)

if RUN_INSTALL:
    print("Python executable:", sys.executable)
    print("Python version:", sys.version)

    pkgs = ["PIL", "pillow", "numpy", "scipy", "torchvision", "vllm", "transformers"]
    run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", *pkgs], check=False)

    # Hard-delete stale directories that can survive partial upgrades.
    for site_dir in site.getsitepackages():
        p = Path(site_dir)
        if not p.exists():
            continue
        for pattern in [
            "PIL", "pillow-*", "Pillow-*",
            "numpy", "numpy-*",
            "scipy", "scipy-*",
            "torchvision", "torchvision-*",
            "vllm", "vllm-*",
            "transformers", "transformers-*",
        ]:
            for target in p.glob(pattern):
                shutil.rmtree(target, ignore_errors=True)

    run([sys.executable, "-m", "pip", "install", "-q", "-U", "uv", "tqdm"])

    constraints = Path("/tmp/vllm_constraints.txt")
    constraints.write_text(
        "pillow==11.3.0\n"
        "numpy==2.3.5\n"
        "scipy==1.16.3\n",
        encoding="utf-8",
    )

    run([
        sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "--force-reinstall",
        "-c", str(constraints),
        "pillow", "numpy", "scipy",
    ])

    run([
        "uv", "pip", "install", "--system",
        "vllm",
        "--torch-backend=auto",
        "-c", str(constraints),
    ])

    # Do not fail notebook setup because unrelated preinstalled packages disagree.
    run([sys.executable, "-m", "pip", "check"], check=False)

# Import sanity check in the current kernel.
import PIL
from PIL import Image, ImageDraw
print("Pillow OK:", PIL.__version__, PIL.__file__)

import numpy as np
import scipy
print("NumPy OK:", np.__version__, np.__file__)
print("SciPy OK:", scipy.__version__, scipy.__file__)

from numpy._core.umath import _center
print("NumPy internal symbol OK")

from vllm import LLM, SamplingParams
print("vLLM import OK")


In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

## 2 · Configuration

In [ ]:
from pathlib import Path

# ── Paths ────────────────────────────────────────────────────────────────────
BASE_DIR     = Path('/content/drive/MyDrive/swiss_law') if IN_COLAB else Path('..').resolve()
DATA_DIR     = BASE_DIR / 'data'
INSIGHTS_DIR = BASE_DIR / 'data_insights'
ART_DIR      = BASE_DIR / 'artifacts'
SCRIPT_DIR   = BASE_DIR / 'scripts'

for d in [DATA_DIR, INSIGHTS_DIR, ART_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Model ────────────────────────────────────────────────────────────────────
# Official post-trained Qwen3.5 35B-A3B MoE checkpoint.
MODEL_ID     = 'Qwen/Qwen3.5-35B-A3B'
QUANTIZATION = None  # official BF16; keep None unless you intentionally switch to a quantized checkpoint

# ── Inference / throughput ───────────────────────────────────────────────────
GPU_MEMORY_UTIL      = 0.90
MAX_MODEL_LEN        = 4096
BATCH_SIZE           = 32       # observed stable on 95.6 GB VRAM; try 64 only after quality is stable
TEMPERATURE          = 0.0      # deterministic extraction
TOP_P                = 1.0
MAX_TOKENS           = 384      # fast first pass
RETRY_MAX_TOKENS     = 768      # used only for invalid/partial JSON retries
TENSOR_PARALLEL      = 1
TEXT_CHARS           = 1800     # truncate source paragraph to bound latency
ENABLE_PREFIX_CACHE  = True     # throughput path; repeated system/template prefix benefits from cache

# ── Quality guardrails ───────────────────────────────────────────────────────
RETRY_BAD_JSON              = True
ABORT_IF_ERROR_RATE_ABOVE   = 0.10   # abort after warmup if >10% parse/validation failures
ERROR_RATE_CHECK_AFTER      = 200    # start checking after this many LLM attempts

# ── Files ────────────────────────────────────────────────────────────────────
INPUT_FILE      = DATA_DIR / 'court_authority_cards_v4.jsonl'
OUTPUT_FILE     = ART_DIR  / 'court_authority_cards_rag.jsonl'
CHECKPOINT_FILE = ART_DIR  / 'rag_checkpoint.txt'

# Optional: stop after N input cards (0 = process all)
LIMIT = 0

# Set True only when you intentionally want a clean re-run from line 0.
RESET_OUTPUT = False

if RESET_OUTPUT:
    OUTPUT_FILE.unlink(missing_ok=True)
    CHECKPOINT_FILE.unlink(missing_ok=True)

print('BASE_DIR   :', BASE_DIR)
print('INPUT_FILE :', INPUT_FILE)
print('OUTPUT_FILE:', OUTPUT_FILE)
print('MODEL_ID   :', MODEL_ID)
print(f'BATCH_SIZE={BATCH_SIZE}, MAX_TOKENS={MAX_TOKENS}, RETRY_MAX_TOKENS={RETRY_MAX_TOKENS}')


## 3 · JSON schema + prompts

In [ ]:
import re

# ── Pre-filter: trivial paragraphs that don't need an LLM ────────────────────
COST_PROC_RE = re.compile(
    r'(?:'
    r'\bgerichtskosten\b|\bprozesskosten\b|\bverfahrenskosten\b|'
    r'\bfrais judiciaires\b|\bfrais de la cause\b|\bd[eé]pens\b|'
    r'\bspese giudiziarie\b|\bripetibili\b|'
    r'\bparteientsch[äa]digung\b|\bhonoraire\b|'
    r'\bunentgeltliche rechtspflege\b|\bassistance judiciaire\b|'
    r'\bpatrocinio gratuito\b|'
    r'\bdie sache wird .{0,80}zur[üu]ckgewiesen\b|'
    r'\brenvoyer la cause\b|'
    r'\bla causa [eè] rinviata\b|'
    r'^\s*\d+\.\s*\d+\..{0,5}fr\.\s*\d'
    r')',
    re.IGNORECASE | re.MULTILINE,
)

# ── JSON schema for constrained generation ───────────────────────────────────
# Keep the schema compact and strict. The method field is added by Python after parsing.
RAG_SCHEMA = {
    'type': 'object',
    'additionalProperties': False,
    'properties': {
        'english_summary':          {'type': 'string'},
        'legal_topic':              {'type': 'string'},
        'legal_question':           {'type': 'string'},
        'legal_rule':               {'type': 'string'},
        'court_holding':            {'type': 'string'},
        'factual_context':          {'type': 'string'},
        'english_legal_concepts':   {'type': 'array', 'items': {'type': 'string'}, 'maxItems': 8},
        'search_keywords':          {'type': 'array', 'items': {'type': 'string'}, 'maxItems': 10},
        'natural_language_queries': {'type': 'array', 'items': {'type': 'string'}, 'maxItems': 5},
        'paragraph_role': {
            'type': 'string',
            'enum': ['holding', 'reasoning', 'background', 'cost',
                     'procedural', 'disposition', 'standard_of_review', 'obiter'],
        },
        'outcome_signal': {
            'type': 'string',
            'enum': ['granted', 'dismissed', 'inadmissible', 'remitted', 'partial', 'none'],
        },
    },
    'required': [
        'english_summary',
        'legal_topic',
        'legal_question',
        'legal_rule',
        'court_holding',
        'factual_context',
        'english_legal_concepts',
        'search_keywords',
        'natural_language_queries',
        'paragraph_role',
        'outcome_signal',
    ],
}

SYSTEM_PROMPT = (
    'You are a deterministic Swiss legal JSON extraction engine. '
    'The user gives one paragraph from a Swiss Federal Tribunal decision in German, French, or Italian. '
    'Return exactly one JSON object matching the provided schema. '
    'Translate concepts into precise English legal terminology. '
    'Prefer concrete legal phrases, e.g. "extension of pretrial detention based on flight risk" rather than "detention". '
    'If a field is not supported by the paragraph, use an empty string, an empty array, or "none" for outcome_signal. '
    'Do not continue the input text. Do not output markdown. Do not output citations or bibliography unless they are part of a field value. '
    'Do not output chain-of-thought.'
)

print('Schema fields:', list(RAG_SCHEMA['properties'].keys()))
print('Required fields:', RAG_SCHEMA['required'])


## 4 · Helper functions

In [ ]:
import json
import re
from pathlib import Path
from typing import Iterator, Any


def build_user_message(card: dict) -> str:
    text = (card.get('text_excerpt_original', '') or '')[:TEXT_CHARS]
    citation = card.get('citation', '') or ''
    legal_area = card.get('legal_area', '') or ''
    existing = card.get('issue_labels_en') or []

    parts = [
        '<record>',
        f'<citation>{citation}</citation>',
        f'<legal_area>{legal_area}</legal_area>',
    ]

    if existing:
        labels = ', '.join(map(str, existing[:8]))
        parts.append(f'<existing_labels>{labels}</existing_labels>')

    parts += [
        '<paragraph_original_language>',
        text,
        '</paragraph_original_language>',
        '</record>',
        '',
        'Return exactly one JSON object matching the schema.',
    ]
    return '\n'.join(parts)


def _stub(summary, topic, concepts, keywords, role, method):
    return {
        'english_summary':          summary,
        'legal_topic':              topic,
        'legal_question':           '',
        'legal_rule':               '',
        'court_holding':            '',
        'factual_context':          '',
        'english_legal_concepts':   concepts,
        'search_keywords':          keywords,
        'natural_language_queries': [],
        'paragraph_role':           role,
        'outcome_signal':           'none',
        'method':                   method,
    }


def auto_classify(card: dict) -> dict | None:
    if card.get('is_notification_paragraph'):
        return _stub(
            'Procedural notification of the judgment to the parties.',
            'judgment notification',
            ['service of judgment'],
            ['notification', 'service', 'judgment communication'],
            role='procedural',
            method='auto_notification',
        )

    text = card.get('text_excerpt_original', '') or ''

    if len(text) < 50:
        return _stub(
            'Short procedural fragment, cross-reference, or one-line ruling.',
            'procedural fragment',
            [], [],
            role='procedural',
            method='auto_short',
        )

    if COST_PROC_RE.search(text[:400]):
        return _stub(
            'Court-cost, procedural-fee, legal-aid, or remittal paragraph.',
            'court costs and procedural fees',
            ['court costs', 'procedural fees', 'legal aid', 'remittal'],
            ['costs', 'court fees', 'frais judiciaires', 'Gerichtskosten', 'legal aid'],
            role='cost',
            method='auto_cost',
        )

    return None


def stream_input(path: Path, start_offset: int) -> Iterator[tuple[int, dict]]:
    with path.open(encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i < start_offset or not line.strip():
                continue
            try:
                yield i, json.loads(line)
            except json.JSONDecodeError:
                continue


def count_lines(path: Path) -> int:
    n = 0
    with path.open('rb') as f:
        for _ in f:
            n += 1
    return n


def render_prompt(card: dict, *, extra_guard: str = '') -> str:
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': build_user_message(card) + extra_guard},
    ]

    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        # Older tokenizer versions may not expose enable_thinking.
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        return prompt + '\nDo not output chain-of-thought. Output only the JSON object.\n'


def clean_model_json(raw: str) -> str:
    raw = (raw or '').strip()

    if '</think>' in raw:
        raw = raw.split('</think>', 1)[1].strip()

    if raw.startswith('```json'):
        raw = raw[7:].strip()
    if raw.startswith('```'):
        raw = raw[3:].strip()
    if raw.endswith('```'):
        raw = raw[:-3].strip()

    # Defensive fallback for non-constrained decoding: extract the outermost object.
    if raw and not raw.startswith('{'):
        start = raw.find('{')
        end = raw.rfind('}')
        if start != -1 and end != -1 and end > start:
            raw = raw[start:end + 1].strip()

    return raw


def validate_and_normalize_enrichment(enriched: Any, *, method: str) -> dict:
    if not isinstance(enriched, dict):
        raise ValueError(f'Expected dict, got {type(enriched).__name__}')

    required = RAG_SCHEMA['required']
    missing = [k for k in required if k not in enriched]
    if missing:
        raise ValueError(f'Missing required fields: {missing}')

    # Normalize scalar fields.
    string_fields = [
        'english_summary', 'legal_topic', 'legal_question',
        'legal_rule', 'court_holding', 'factual_context',
    ]
    for key in string_fields:
        value = enriched.get(key, '')
        enriched[key] = value if isinstance(value, str) else str(value)

    # Normalize arrays.
    for key, max_items in [
        ('english_legal_concepts', 8),
        ('search_keywords', 10),
        ('natural_language_queries', 5),
    ]:
        value = enriched.get(key, [])
        if not isinstance(value, list):
            value = [str(value)] if value else []
        enriched[key] = [str(x).strip() for x in value if str(x).strip()][:max_items]

    if enriched.get('paragraph_role') not in RAG_SCHEMA['properties']['paragraph_role']['enum']:
        enriched['paragraph_role'] = 'reasoning'

    if enriched.get('outcome_signal') not in RAG_SCHEMA['properties']['outcome_signal']['enum']:
        enriched['outcome_signal'] = 'none'

    enriched['method'] = method
    return enriched



def parse_output_text(raw_text: str, *, method: str) -> dict:
    """Clean, parse, validate, and normalize one model output."""
    raw = clean_model_json(raw_text)
    enriched = json.loads(raw)
    return validate_and_normalize_enrichment(enriched, method=method)


print('Helpers loaded.')


## 5 · Load model

> **Note:** Loading Qwen3.5-35B-A3B in BF16 downloads ~70 GB from HuggingFace Hub on first run.  
> Subsequent runs load from the Colab disk cache (typically `/root/.cache/huggingface`).
>
> **Thinking mode is explicitly disabled** via `chat_template_kwargs={"enable_thinking": False}`.  
> Without this, the model emits `<think>…</think>` tokens that break guided JSON decoding.

In [ ]:
import os
import inspect

# Must be set before initializing vLLM.
# DEBUG avoids a Jupyter/ipykernel sys.stdout.fileno() crash inside vLLM stdout suppression.
os.environ['VLLM_LOGGING_LEVEL'] = 'DEBUG'

# Keeps the vLLM V1 engine in-process in notebooks and exposes real tracebacks.
os.environ['VLLM_ENABLE_V1_MULTIPROCESSING'] = '0'

# Remove stale/invalid env vars from prior experiments.
os.environ.pop('VLLM_WORKER_MULTIPROC_METHOD', None)
os.environ.pop('VLLM_MOE_BACKEND', None)
os.environ.pop('VLLM_FLASHINFER_MOE_BACKEND', None)

from vllm import LLM, SamplingParams
from vllm.config import KernelConfig

try:
    from vllm.sampling_params import StructuredOutputsParams
except Exception:
    StructuredOutputsParams = None

try:
    from vllm.sampling_params import GuidedDecodingParams
except Exception:
    GuidedDecodingParams = None


def build_llm_kwargs():
    kwargs = dict(
        model=MODEL_ID,
        quantization=QUANTIZATION,
        dtype='bfloat16',
        gpu_memory_utilization=GPU_MEMORY_UTIL,
        max_model_len=MAX_MODEL_LEN,
        tensor_parallel_size=TENSOR_PARALLEL,
        trust_remote_code=False,
        enforce_eager=True,
        enable_prefix_caching=ENABLE_PREFIX_CACHE,
        disable_log_stats=True,
        limit_mm_per_prompt={'image': 0, 'video': 0},

        # Blackwell fix: force Triton MoE so vLLM does not auto-select
        # FlashInfer CUTLASS MoE, which can fail on SM120 if kernels/JIT cache are not ready.
        kernel_config=KernelConfig(moe_backend='triton'),

        # Throughput hint for offline batched generation.
        max_num_seqs=max(BATCH_SIZE, 64),
    )

    sig = inspect.signature(LLM.__init__)
    if 'language_model_only' in sig.parameters:
        kwargs['language_model_only'] = True

    return kwargs


def make_sampling_params(schema, *, max_tokens: int):
    base = dict(
        temperature=TEMPERATURE,
        top_p=TOP_P,
        max_tokens=max_tokens,
    )

    # Prefer current structured output API.
    if StructuredOutputsParams is not None:
        try:
            params = SamplingParams(
                **base,
                structured_outputs=StructuredOutputsParams(json=schema),
            )
            print(f'Using StructuredOutputsParams(json=...) with max_tokens={max_tokens}')
            return params
        except TypeError as e:
            print('StructuredOutputsParams failed; trying GuidedDecodingParams:', repr(e))

    # Fallback for older vLLM versions.
    if GuidedDecodingParams is not None:
        params = SamplingParams(
            **base,
            guided_decoding=GuidedDecodingParams(json=schema),
        )
        print(f'Using GuidedDecodingParams(json=...) with max_tokens={max_tokens}')
        return params

    raise RuntimeError('No vLLM structured JSON decoding support found. Do not run unconstrained generation.')


print('Loading Qwen3.5 with vLLM offline inference...')
print(f'MODEL_ID={MODEL_ID}')
print(
    f'max_model_len={MAX_MODEL_LEN}, gpu_memory_utilization={GPU_MEMORY_UTIL}, '
    f'batch_size={BATCH_SIZE}, max_tokens={MAX_TOKENS}, retry_max_tokens={RETRY_MAX_TOKENS}'
)

llm = LLM(**build_llm_kwargs())
tokenizer = llm.get_tokenizer()

sampling_params = make_sampling_params(RAG_SCHEMA, max_tokens=MAX_TOKENS)
retry_sampling_params = make_sampling_params(RAG_SCHEMA, max_tokens=RETRY_MAX_TOKENS)

print('Sampling params:', sampling_params)
print('Model loaded successfully.')


## 5.5 · Optional smoke test

Run this before the full enrichment cell if you changed model or schema settings. It tests a small number of non-auto cards without writing output.


In [ ]:
# Optional smoke test: set RUN_SMOKE_TEST = True and run this cell.
RUN_SMOKE_TEST = False
SMOKE_N = 8

if RUN_SMOKE_TEST:
    samples = []
    for _, card in stream_input(INPUT_FILE, 0):
        if auto_classify(card) is None:
            samples.append(card)
        if len(samples) >= SMOKE_N:
            break

    prompts = [render_prompt(card) for card in samples]
    outs = llm.generate(prompts, sampling_params=sampling_params, use_tqdm=False)

    failures = 0
    for i, (card, out) in enumerate(zip(samples, outs)):
        raw = out.outputs[0].text
        print('\n' + '=' * 80)
        print('SAMPLE', i, '| citation:', card.get('citation', ''))
        try:
            obj = parse_output_text(raw, method='smoke')
            print(json.dumps(obj, ensure_ascii=False, indent=2)[:1200])
        except Exception as e:
            failures += 1
            print('FAILED:', repr(e))
            print(clean_model_json(raw)[:1200])

    if failures:
        raise RuntimeError(f'Smoke test failed: {failures}/{len(samples)} invalid JSON outputs.')
    print(f'Smoke test passed: {len(samples)} valid JSON outputs.')


## 6 · Run enrichment

In [ ]:
from tqdm.auto import tqdm
import time

if not INPUT_FILE.exists():
    raise FileNotFoundError(f'Input not found: {INPUT_FILE}')

start = 0
if CHECKPOINT_FILE.exists():
    try:
        start = int(CHECKPOINT_FILE.read_text().strip() or '0')
    except ValueError:
        start = 0

print(f'Resuming at line {start:,}')
print(f'Counting lines in {INPUT_FILE.name} ...')

total_lines = count_lines(INPUT_FILE)
target_total = min(total_lines, start + LIMIT) if LIMIT else total_lines

print(f'Total={total_lines:,}  To process={target_total - start:,}')
print(f'Using BATCH_SIZE={BATCH_SIZE}, MAX_TOKENS={MAX_TOKENS}, RETRY_MAX_TOKENS={RETRY_MAX_TOKENS}')

out_f = OUTPUT_FILE.open('a', encoding='utf-8')
pbar = tqdm(total=target_total, initial=start, desc='enrich', unit='card', smoothing=0.03)

pending: list[tuple[int, dict]] = []
json_errors = 0
llm_attempts = 0
batch_count = 0
started_at = time.time()


def parse_output_text(raw_text: str, *, method: str) -> dict:
    raw = clean_model_json(raw_text)
    enriched = json.loads(raw)
    return validate_and_normalize_enrichment(enriched, method=method)


def retry_one(card: dict) -> tuple[dict, str | None]:
    retry_guard = (
        '\n\nPrevious generation was invalid. '
        'Return one complete JSON object only. '
        'No prose, no markdown, no citations outside JSON fields.'
    )
    prompt = render_prompt(card, extra_guard=retry_guard)
    out = llm.generate([prompt], sampling_params=retry_sampling_params, use_tqdm=False)[0]
    raw = out.outputs[0].text

    try:
        return parse_output_text(raw, method='qwen35_35b_a3b_vllm_retry'), None
    except Exception as e:
        return (
            _stub('', '', [], [], role='reasoning', method='json_parse_failed'),
            f'{type(e).__name__}: {str(e)[:200]} | raw={clean_model_json(raw)[:300]}',
        )


def flush_batch():
    global pending, json_errors, llm_attempts, batch_count

    if not pending:
        return

    batch_count += 1
    batch_size_now = len(pending)
    prompts = [render_prompt(card) for _, card in pending]

    t0 = time.time()
    outputs = llm.generate(prompts, sampling_params=sampling_params, use_tqdm=False)
    dt = time.time() - t0

    llm_attempts += batch_size_now
    batch_json_errors = 0

    for (line_idx, card), out in zip(pending, outputs):
        raw_text = out.outputs[0].text

        try:
            enriched = parse_output_text(raw_text, method='qwen35_35b_a3b_vllm')
        except Exception as first_error:
            if RETRY_BAD_JSON:
                enriched, retry_error = retry_one(card)
                llm_attempts += 1
                if retry_error is not None:
                    json_errors += 1
                    batch_json_errors += 1
                    enriched['parse_error'] = retry_error
                    enriched['raw_output'] = clean_model_json(raw_text)[:400]
            else:
                json_errors += 1
                batch_json_errors += 1
                enriched = _stub('', '', [], [], role='reasoning', method='json_parse_failed')
                enriched['parse_error'] = f'{type(first_error).__name__}: {str(first_error)[:200]}'
                enriched['raw_output'] = clean_model_json(raw_text)[:400]

        card['rag_enrichment'] = enriched
        out_f.write(json.dumps(card, ensure_ascii=False) + '\n')

    out_f.flush()
    CHECKPOINT_FILE.write_text(str(pending[-1][0] + 1))
    pbar.update(len(pending))

    elapsed = time.time() - started_at
    done = max(pbar.n - start, 0)
    remaining = max(target_total - pbar.n, 0)
    avg_rate = done / max(elapsed, 1e-9)
    batch_rate = batch_size_now / max(dt, 1e-9)
    eta_min = remaining / max(avg_rate, 1e-9) / 60.0

    if batch_count == 1 or batch_count % 5 == 0 or batch_json_errors:
        print(
            f'[batch {batch_count}] size={batch_size_now}, '
            f'batch_time={dt:.1f}s, batch_rate={batch_rate:.2f} cards/s, '
            f'avg_rate={avg_rate:.2f} cards/s, eta≈{eta_min:.1f} min, '
            f'batch_json_errors={batch_json_errors}, total_json_errors={json_errors}, '
            f'llm_attempts={llm_attempts}'
        )

    if llm_attempts >= ERROR_RATE_CHECK_AFTER:
        err_rate = json_errors / max(llm_attempts, 1)
        if err_rate > ABORT_IF_ERROR_RATE_ABOVE:
            raise RuntimeError(
                f'Aborting: JSON error rate {err_rate:.1%} exceeds '
                f'{ABORT_IF_ERROR_RATE_ABOVE:.1%}. Inspect failed raw outputs before scaling.'
            )

    pending.clear()


processed = 0

try:
    for line_idx, card in stream_input(INPUT_FILE, start):
        if LIMIT and processed >= LIMIT:
            break

        auto = auto_classify(card)
        if auto is not None:
            card['rag_enrichment'] = auto
            out_f.write(json.dumps(card, ensure_ascii=False) + '\n')
            CHECKPOINT_FILE.write_text(str(line_idx + 1))
            pbar.update(1)
            processed += 1
            continue

        pending.append((line_idx, card))

        if len(pending) >= BATCH_SIZE:
            flush_batch()

        processed += 1

    flush_batch()

finally:
    out_f.close()
    pbar.close()

print(f'Done. JSON parse/validation errors after retry: {json_errors}')
print(f'Output → {OUTPUT_FILE}')


## 7 · Verify output

In [ ]:
from collections import Counter
import json

roles = Counter()
outcomes = Counter()
methods = Counter()
total_out = 0
missing_required = 0
bad = []

REQUIRED = RAG_SCHEMA['required']

if not OUTPUT_FILE.exists():
    raise FileNotFoundError(f'Output not found: {OUTPUT_FILE}')

with OUTPUT_FILE.open(encoding='utf-8') as f:
    for line in f:
        if not line.strip():
            continue

        card = json.loads(line)
        e = card.get('rag_enrichment', {}) or {}
        total_out += 1

        roles[e.get('paragraph_role', 'MISSING')] += 1
        outcomes[e.get('outcome_signal', 'MISSING')] += 1
        methods[e.get('method', 'MISSING')] += 1

        if any(k not in e for k in REQUIRED):
            missing_required += 1
            if len(bad) < 10:
                bad.append(card)

        if e.get('method') == 'json_parse_failed' and len(bad) < 10:
            bad.append(card)

print(f'Total output cards : {total_out:,}')
print(f'Missing required   : {missing_required:,}')
print(f'Failed JSON method : {methods.get("json_parse_failed", 0):,}')
print()

if total_out:
    print('paragraph_role distribution:')
    for k, v in roles.most_common():
        print(f'  {k:<25} {v:>8}  ({v/total_out*100:.1f}%)')

    print()
    print('outcome_signal distribution:')
    for k, v in outcomes.most_common():
        print(f'  {k:<25} {v:>8}  ({v/total_out*100:.1f}%)')

    print()
    print('method distribution:')
    for k, v in methods.most_common():
        print(f'  {k:<30} {v:>8}  ({v/total_out*100:.1f}%)')

if bad:
    print('\nExamples needing inspection:')
    for i, card in enumerate(bad[:5]):
        e = card.get('rag_enrichment', {})
        print('\n--- BAD', i, '---')
        print('citation:', card.get('citation', ''))
        print('method:', e.get('method'))
        print('parse_error:', e.get('parse_error'))
        print('raw_output:', (e.get('raw_output') or '')[:500])


In [ ]:
# Show a few LLM-enriched cards for a quick quality check.
import random
import json

llm_cards = []
with OUTPUT_FILE.open(encoding='utf-8') as f:
    for line in f:
        if not line.strip():
            continue
        card = json.loads(line)
        method = card.get('rag_enrichment', {}).get('method', '')
        if method.startswith('qwen35') and method != 'json_parse_failed':
            llm_cards.append(card)

print(f'LLM-enriched valid cards available: {len(llm_cards):,}')

for card in random.sample(llm_cards, min(3, len(llm_cards))):
    e = card['rag_enrichment']
    print('─' * 90)
    print('Citation     :', card.get('citation', ''))
    print('Role         :', e.get('paragraph_role'))
    print('Outcome      :', e.get('outcome_signal'))
    print('Topic        :', e.get('legal_topic'))
    print('Question     :', e.get('legal_question', '')[:240])
    print('Rule         :', e.get('legal_rule', '')[:240])
    print('Holding      :', e.get('court_holding', '')[:240])
    print('Summary      :', e.get('english_summary', '')[:240])
    print('Concepts     :', e.get('english_legal_concepts'))
    print('Keywords     :', e.get('search_keywords'))
    print('NL queries   :', e.get('natural_language_queries'))
